# Example Usage of SpliceSeqExtractor Class

Below is a very basic workflow using the SpliceSeqExtractor class. We will import the class, provide the necessary paths, and then observe the output data structures.

In [ ]:
from splice_seq_extractor import SpliceSeqExtractor
from itertools import islice

For the data, I used gencode v48 human, chromosome 1. This class should work with any GFF3 and FASTA from any species, provided that you're using the same version numbers for both files (else the annotations won't match the proper coordinates). Of course, annotation quality is going to vary between species, especially for those that aren't very well-studied. GENCODE doesn't provide per-chromosome annotations as separate files, so I will include some simple bash scripts that will split out the whole genome by chromosome, located in `protisplice/bash_utils`.

First, we can instantiate the class. Note that the gff path and FASTA paths are required arguments, and the transcript filter is optional. By default, the transcript filter is `"all"`, but we can filter by transcripts that are considered protein coding by the annotation file with `"protein_coding"`.


In [ ]:
extractor = SpliceSeqExtractor(gff_path="data/input/chr1.gff3", fasta_path="data/input/chr1.fa", transcript_filter="protein_coding")


By default, the number of exon bases to include is 40, and 80 for intron bases (making a window size of 120). We can double check the parameters provided with `get_info()`:

In [ ]:
extractor.get_info()

The extractor has two properties - `junctions` and `transcripts`. These are lazy-loaded, so they will be computed upon any method that requires these properties. These properties are also cached as well. First, we can take a look at the identified junctions:

In [ ]:
junctions = extractor.junctions

The junctions are generated using a `SpliceJunction` object, which essentially is just a dictionary containing the transcript ID, sequence ID (usually going to be the chromosome # for most purposes), the coordinate of the junction, strand, and the type of junction - either donor or acceptor.

Note that the junction coordinates are 1-based and represent the first exon base along the junction (i.e. coord - 1 is the end of the intron). 

Currently there is a big limitation in the determination of donor/acceptor sites in that they don't take alternative splicing into account. Later I might find some sort of way to correct this using an alt splicing DB or RNA-seq data. For now, just note that the first exon of a transcript is always only a donor, the last one is only an acceptor, and anything in between can be either donor or acceptor. 

Lastly, the transcript ID is suffixed with ID_junctype_num, where junctype is either donor/acceptor, and num is the sequential order in which each splice junction is found, starting at 0. 

In [ ]:
junctions[:5]

We can also take a look at all of the transcripts found using `extractor.transcripts`, which will return a `Transcript` object, a dictionary of dictionaries. `Transcript.info` contains the sequence ID from the associated FASTA and the strand type. 

`Transcript.exons` contains the 1-based start and end coordinates for every exon in the transcript. `Transcript.introns` is initiated to `None` but can be determined using `Transcript.get_intron_coords`. 

Reason for this being that the introns can already be inferred from the exon coordinates, and it would take up more memory to explicitly hold the intron coordinates as well, but the method exists if you need it.

In [ ]:
transcripts = extractor.transcripts
dict(islice(transcripts.items(), 5))

Lastly, we will get to the bigger workflow functions from this class. `extract_splice_sites` will return a JunctionData object containing the splice junction, window start, window end, and the sequence.

In [ ]:
true_ss = extractor.sequence_extractor.extract_splice_sites(junctions)
true_ss[:5]

In [ ]:
false_ss = extractor.sequence_extractor.sample_introns(transcripts=transcripts, target_count=10000)
false_ss[:5]